# **AfriLink SDK — End-to-End Finetune Demo (OpenToken A100 Backend)**

This notebook walks through the **complete workflow** of the AfriLink SDK against the **OpenToken A100** backend — a dedicated NVIDIA A100 80 GB compute node hosted by [OpenToken](https://opentoken.global). As of SDK **v0.8.x** all jobs run as ephemeral Docker containers on the A100: no SLURM, no 12-hour SSH certificate, no email/password prompts. Auth is a single notebook secret (`AFRILINK_API_KEY`) minted from the DataSpires dashboard.

| Step | What | API |
|------|------|-----|
| 1 | Install SDK | `pip install 'afrilink-sdk[build]'` |
| 2 | Authenticate (stateless API key) | `client.authenticate()` |
| 3 | Browse available models & datasets | `client.list_available_models()` |
| 4 | Prepare dataset | pandas DataFrame |
| 5 | Submit finetune job on the OpenToken A100 | `client.finetune(...).run(wait=True)` |
| 6 | Download trained weights | `client.download_model(job_id, dir)` |
| 7 | Load adapter & test your model | `PeftModel.from_pretrained(base, dir)` |
| 8 | **NEW** Reuse-or-build custom containers | `client.find_existing_image(...)` + `client.build_and_train(...)` |
| 9 | Deploy locally with GGUF & Ollama | offline inference, no GPU needed |

**Prerequisites:**
- A free [DataSpires](https://dataspires.com) account
- An AfriLink API key minted at [dataspires.com/dashboard/profile](https://dataspires.com/dashboard/profile) → **AfriLink SDK keys** → **Create new key**
- The key set as a notebook secret named exactly `AFRILINK_API_KEY` (Colab: 🔑 sidebar → Add secret; Kaggle: Add-ons → Secrets)

You can run the cells with the example inputs to see each path in action, or swap in your own model / dataset / container spec.

---
## 1. **Install the AfriLink SDK**

AfriLink SDK gives you one-line access to a dedicated NVIDIA A100 80 GB for **training and finetuning** across text, vision and multimodal models. Works on **Google Colab, Kaggle, Jupyter, VS Code**, and any Python environment.

Install with the `[build]` extras to get the deps for the custom-container path used later in §8 (`cryptography` for signing the GCP service-account JWT, `requests` for the Cloud Build REST calls).

Just run `pip install 'afrilink-sdk[build]>=0.8.12'` and follow the rest of the steps below!

In [ ]:
!pip install -q -U 'afrilink-sdk[build]>=0.8.15'

import afrilink
print(f"AfriLink SDK v{afrilink.__version__} ready")


---
## 2. **Authenticate with your AfriLink API Key**

A single `client.authenticate()` call handles everything:

1. **DataSpires login** — the SDK reads `AFRILINK_API_KEY` from your notebook secrets and exchanges it for a short-lived session token used for billing writes.
2. **OpenToken A100 reachability** — a silent SSH probe to the A100 to confirm your slot is live.

Expect this cell to finish in ~2 seconds. No prompts, no SSH key files, no 12-hour cert refresh.

In [ ]:
from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()   # uses AFRILINK_API_KEY from notebook secrets


In [ ]:
# Quick sanity check that the OpenToken A100 is reachable and the GPU is visible.
code, out, err = client.run_command(
    'nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader'
)
print(out)


---
## 3. **Browse AfriLink's Available Models & Datasets**

The SDK ships with a registry of pre-configured models (text LLMs and vision-language models) and datasets.

For this demo, the model that's ready for full testing is the `qwen2.5-0.5b` model — Apache 2.0, no licensing wall, fits comfortably on the A100.

In [4]:
from afrilink import list_models, list_datasets

models = list_models()

print("TEXT MODELS")
print("-" * 65)
for m in models:
    if m["type"] == "text":
        print(f"  {m['id']:20} {m['parameters_b']:.2f}B  {m['min_gpu_memory_gb']:>2}GB VRAM  {m['name']}")

print("\nVISION-LANGUAGE MODELS")
print("-" * 65)
for m in models:
    if m["type"] == "vision":
        print(f"  {m['id']:20} {m['parameters_b']:.2f}B  {m['min_gpu_memory_gb']:>2}GB VRAM  {m['name']}")

print("\nDATASETS")
print("-" * 65)
for d in list_datasets():
    n = d.get('num_examples') or 'N/A'
    print(f"  {d['id']:25} {str(n):>8} examples  {d['name']}")

TEXT MODELS
-----------------------------------------------------------------
  qwen2.5-0.5b         0.50B   4GB VRAM  Qwen 2.5 0.5B
  gemma-3-270m         0.27B   2GB VRAM  Gemma 3 270M
  ministral-3b         3.30B   8GB VRAM  Ministral 3B Reasoning
  llama-3.2-1b         1.00B   4GB VRAM  Llama 3.2 1B
  deepseek-r1-1.5b     1.50B   6GB VRAM  DeepSeek R1 Distill Qwen 1.5B

VISION-LANGUAGE MODELS
-----------------------------------------------------------------
  llava-1.5-7b         7.00B  16GB VRAM  LLaVA 1.5 7B
  moondream2           1.90B   8GB VRAM  Moondream 2
  florence-2-base      0.23B   4GB VRAM  Florence 2 Base
  smolvlm-256m         0.26B   2GB VRAM  SmolVLM 256M Instruct
  internvl2-1b         1.00B   4GB VRAM  InternVL2 1B

DATASETS
-----------------------------------------------------------------
  multilingual-thinking          N/A examples  Multilingual Thinking
  alpaca                       52000 examples  Stanford Alpaca
  dolly                        15000 examples

In [5]:
# Check resource requirements for a specific model
client.get_model_requirements("qwen2.5-0.5b", "low")

{'model': 'qwen2.5-0.5b',
 'model_type': 'text',
 'training_mode': 'low',
 'recommended_gpus': 1,
 'min_memory_gb': 4,
 'parameters_b': 0.5,
 'context_length': 32768,
 'requires_vision_encoder': False}

---
## 4. **Prepare a Small Example Dataset**

The SDK accepts all these kinds of dataset configurations:
- **pandas DataFrame** with a `text` column (Alpaca-style prompt+response in one string)
- **HuggingFace Dataset**
- **File path** to a local JSONL/CSV

Below we create a small example. In production you'd use a real dataset loaded from your notebook with `pandas`.

In [6]:
import pandas as pd

# Alpaca-style: each row is a self-contained prompt + response
rows = [
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is machine learning?\n\n"
    "### Response:\nMachine learning is a branch of artificial intelligence that "
    "enables systems to learn patterns from data and improve with experience.",

    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nExplain gradient descent in one sentence.\n\n"
    "### Response:\nGradient descent is an optimisation algorithm that iteratively "
    "adjusts model parameters in the direction that minimises the loss function.",

    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is a neural network?\n\n"
    "### Response:\nA neural network is a computing architecture composed of "
    "interconnected layers of nodes that learns to map inputs to outputs.",
]

dataset = pd.DataFrame({"text": rows})
print(f"Dataset: {len(dataset)} rows")
print(dataset.head())

Dataset: 3 rows
                                                text
0  Below is an instruction that describes a task....
1  Below is an instruction that describes a task....
2  Below is an instruction that describes a task....


---
## 5. **Submit Finetune Job to the OpenToken A100**

`client.finetune()` creates the job; `.run(wait=True)` submits it and blocks until completion.

Behind the scenes the SDK:
1. Serialises your DataFrame to JSONL and SCPs it to `/mnt/data/sdk-jobs/<job_id>/input/` on the A100
2. Submits the curated `afrilink-finetune` container with `docker run --gpus all` and polls until completion
3. Bills wall-clock × $2.00/GPU-hour (1-minute floor) via the canonical `deduct_credits` RPC — visible on your DataSpires Billing dashboard

The job specification pipeline needs values for these variables:

```python
job = client.finetune(
    model="qwen2.5-0.5b",    # The model you choose to finetune
    training_mode="low",     # How much training you want to do, chosen between low, mid and high
    data=dataset,            # What dataset you chose or prepare in the notebook for the finetune step
    gpus=1,                  # The A100 has 1 GPU; gpus>1 silently clamps to 1
    time_limit="01:00:00",   # The maximum time your job should run for
)
```

In [7]:
# Create and inspect the job (does not submit yet)
job = client.finetune(
    model="qwen2.5-0.5b",
    training_mode="low",     # QLoRA, 4-bit, rank 8
    data=dataset,
    gpus=1,
    time_limit="01:00:00",
)

cfg = job.spec.training_config
print(f"Job ID:       {job.job_id}")
print(f"Model:        {job.spec.model}")
print(f"Mode:         {job.spec.training_mode.value}")
print(f"GPUs:         {job.spec.gpus}")
print(f"LoRA rank:    {cfg.lora_r}")
print(f"Quantisation: {cfg.quantization_bits}-bit" if cfg.use_quantization else "Quantisation: off")
print(f"Batch size:   {cfg.batch_size}")
print(f"LR:           {cfg.learning_rate}")

Job ID:       c8cd4460
Model:        qwen2.5-0.5b
Mode:         low
GPUs:         1
LoRA rank:    8
Quantisation: 4-bit
Batch size:   2
LR:           0.0002


In [ ]:
# Submit to the OpenToken A100 and wait for completion
result = job.run(wait=True, poll_interval=10)

print(f"\nResult: {result}")


In [9]:
# (Optional) Check logs while waiting or after completion
print(job.get_logs(tail=50))

Job Configuration:
{
  "model_path": "/workspace/models/qwen2.5-0.5b",
  "dataset_path": "/workspace/data/train.jsonl",
  "output_dir": "/workspace/output",
  "lora_r": 8,
  "lora_alpha": 16,
  "batch_size": 2,
  "gradient_accumulation_steps": 8,
  "num_epochs": 3,
  "learning_rate": 0.0002
}

Loading model from: /workspace/models/qwen2.5-0.5b
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

Loading dataset from: /workspace/data/train.jsonl

Starting training...
{'train_runtime': 3.2102, 'train_samples_per_second': 2.804, 'train_steps_per_second': 0.935, 'train_loss': 0.5018122990926107, 'epoch': 3.0}

Saving model to: /workspace/output

Training complete!

Job completed at Tue Apr 14 11:13:32 CEST 2026



---
## 6. **Download Trained Model Weights**

The adapter files (`adapter_model.safetensors`, `adapter_config.json`, tokenizer files) get downloaded into the target directory — ready for `PeftModel.from_pretrained()`.

Use `client.download_model(result["job_id"], MODEL_DIR)` with a pre-specified directory for where you'd like the model files to be saved in your notebook's file system. The extra steps here are just to print out status logs.

**Note:** the A100 backend's job output lands at `/mnt/data/sdk-jobs/<job_id>/output/` and `download_model` SCPs that folder wholesale. The adapter files are inside the `output/` subdirectory of the local destination.

In [ ]:
if result.get("status") != "completed":
    print(f"Job did not complete successfully (status: {result.get('status')})")
    print("Check logs with: job.get_logs()")
    if result.get("error"):
        print(f"\nError output:\n{result['error'][:500]}")
else:
    MODEL_DIR = f"./my-model-{result['job_id']}"

    client.download_model(result["job_id"], MODEL_DIR)

    # download_model SCPs the remote /workspace/job/output/ folder wholesale.
    # Adapter files land at <MODEL_DIR>/output/<file>.
    import os
    for root, _, files in os.walk(MODEL_DIR):
        for f in sorted(files):
            full = os.path.join(root, f)
            size = os.path.getsize(full)
            print(f"  {full:60} {size/1024:.1f} KB")

    # Most steps below assume MODEL_DIR points at the dir containing the adapter files.
    # If they're in MODEL_DIR/output/, redirect MODEL_DIR there for the rest of the notebook.
    candidate = os.path.join(MODEL_DIR, "output")
    if os.path.isdir(candidate):
        MODEL_DIR = candidate
        print(f"\nUsing MODEL_DIR = {MODEL_DIR}")


---
## 7. **Load Adapter & Test your trained model**

Load the base model, attach the LoRA adapter, and generate text. You can keep experimenting at this point, or finetune more models!

In [ ]:
import sys
sys.modules["torchao"] = None

from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch
import os

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
MERGED_DIR = os.path.join(MODEL_DIR, "merged")

print("Merging LoRA weights into base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, MODEL_DIR)

model = model.merge_and_unload()

model.save_pretrained(MERGED_DIR)

print(f"Merged model saved to: {MERGED_DIR}")

MODEL_DIR = MERGED_DIR

In [ ]:
import torch
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

# Always load tokenizer from base model (NOT merged dir)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Fix missing pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompt = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n What is the role of High Performance Compute in creating compute sovereignity in Africa?.\n\n### Response:\n"
)

inputs = tokenizer(prompt, return_tensors="pt")

# Guard against empty input (debug safety)
if inputs["input_ids"].shape[1] == 0:
    raise RuntimeError("Tokenizer produced empty input — tokenizer mismatch.")

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Below is an instruction that describes a task.

### Instruction:
What is transfer learning?

### Response:
Transfer learning is a machine learning technique that involves training a model on a dataset that has already been trained on another, related task. By learning from the already trained model, the new model can better generalize and perform well on new, related tasks.

Transfer learning is particularly useful in situations where the dataset for training the model is too large or expensive to generate from scratch. Instead, the model can be trained on a smaller, related dataset, which is usually available either through a dataset repository or a community of researchers. By leveraging the existing model, the new model can learn from the learned knowledge and improve the performance on the new task.

Transfer learning


---
## Utility: **Job Status & Container Management**

Come back to this code cell to check the list of SDK-managed containers on the OpenToken A100 using `client.run_command(...)`.

In [ ]:
# What's running on your A100 slot right now?
code, out, _ = client.run_command(
    "docker ps --filter label=afrilink.managed=true "
    "--format '{{.Names}} {{.Status}} {{.Label \"afrilink.job\"}}'"
)
print("Running AfriLink containers on the A100:")
print(out.strip() or "  (none — your slot is idle)")

# Cancel a job (uncomment and set ID):
# client.cancel_job("ft-XXXXXXXX")


---
## 8. **Reuse-or-Build Custom Containers**

If the curated `afrilink-finetune` container doesn't have what you need — a different base image, extra apt/pip dependencies, a different model — you can define your own. `client.build_image()` and `client.build_and_train()` package your spec, build it on Google Cloud Build (managed, serverless), push it to a private Artifact Registry, and run it on the A100.

**New in v0.8.12:** before triggering a fresh build (~3–5 min) the SDK can check whether a matching image already exists — either pulled onto the A100 from a previous run, or sitting in Artifact Registry from a teammate's session. Use `client.find_existing_image(...)` for an explicit lookup, or pass `reuse_existing_image=True` (the default) to `build_and_train()` and the SDK will short-circuit automatically on a cache hit.

The match is determined by a stable hash of the **build-defining inputs only**:

| Hash includes | Hash excludes |
|---|---|
| `base_image` (after preset resolution) | `script` / `script_content` (uploaded into the image but doesn't change what's baked) |
| `pip_packages` (sorted, exact strings) | `env` (runtime injection, not bake-time) |
| `apt_packages` (sorted, exact strings) | `extra_files` |
| `pip_index_url` / `pip_extra_index_urls` | `user_id` / `job_id` |
| `model_source` (kind + id/url/uri + revision + subfolder) | Cloud Build `machine_type` / `build_timeout` |

Two specs that produce a runtime-equivalent image hash the same → you skip the build. A version bump on any pip package, an extra apt dependency, or a new model revision produces a fresh hash → fresh build.

### 8.1 Check before you build

Define the spec you want, then ask the SDK if it's already available.

In [ ]:
# Define the build spec you want (same args as client.build_image / build_and_train)
spec = dict(
    base_image="pytorch",  # preset: pytorch/pytorch:2.5.0-cuda12.4-cudnn9-runtime
    pip_packages=["transformers>=4.45", "accelerate>=0.34", "peft>=0.13"],
    apt_packages=["git"],
    model_source={
        "kind": "huggingface",
        "id": "Qwen/Qwen2.5-0.5B-Instruct",
        "revision": "main",
    },
)

hit = client.find_existing_image(**spec)
if hit:
    print(f"Cache hit ({hit['source']}): {hit['image']}")
    print(f"Spec hash: {hit['spec_hash']}")
    print("\nYou can pass image=... to a follow-up call to skip the build entirely,")
    print("or just call client.build_and_train(...) with reuse_existing_image=True")
    print("(the default) and the SDK will detect the same cache hit for you.")
else:
    print(f"No cached image for this spec — first build will take ~3–5 min.")


### 8.2 Build-and-train with automatic reuse

`client.build_and_train()` is the convenience wrapper. With `reuse_existing_image=True` (default) it auto-skips the build on cache hits — so the second run of any given spec lands almost instantly on the A100, just paying for the GPU-minutes the container actually uses.

Below: same spec, plus a tiny training script. The first time you run this cell it builds (~3–5 min); every time after that it short-circuits.

In [ ]:
# A minimal training script. In production yours would do real work.
TRAIN_SCRIPT = """
import os, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

models_dir = os.environ.get("MODELS_DIR", "/workspace/models")
print("MODELS_DIR=", models_dir)
print("available:", os.listdir(models_dir))

# The entrypoint fetched the HF model for us
model_path = os.path.join(models_dir, "Qwen__Qwen2.5-0.5B-Instruct")
tok = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")
print(f"Loaded {sum(p.numel() for p in model.parameters())/1e9:.2f}B-param model")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Whatever finetuning you'd normally do goes here. This demo proves the
# model is alive by generating a single token.
out = model.generate(**tok("Hello,", return_tensors="pt").to(model.device), max_new_tokens=10)
sample = tok.decode(out[0], skip_special_tokens=True)
print("sample:", sample)

out_dir = "/workspace/job/output"
os.makedirs(out_dir, exist_ok=True)
with open(f"{out_dir}/result.json", "w") as f:
    json.dump({"ok": True, "sample": sample}, f)
print("done")
"""

with open("my_train.py", "w") as f:
    f.write(TRAIN_SCRIPT)

# build_and_train: builds if needed, runs on the A100, downloads logs,
# wipes the on-A100 image layer (cleanup_image_after=True default).
# The first call builds (~3–5 min). The second call with the same spec
# hits the cache and skips straight to running on the A100.
result = client.build_and_train(
    script="my_train.py",
    gpus=1,
    time_limit_hours=0.5,
    reuse_existing_image=True,   # default; set False to force a fresh build
    **spec,
)

print("BUILD :", result["build"]["image"])
print("       status:", result["build"]["status"])  # 'success' (fresh) or 'cached' (reused)
print("RUN   :", result["run"]["status"], "exit", result["run"]["exit_code"])
print()
print("--- container logs (tail 60) ---")
print(result["run"]["logs_tail"])


### 8.3 Pull the artefact your script wrote

Same `client.download_model(job_id, local_dir)` API as the curated path.

In [ ]:
client.download_model(result["run"]["job_id"], "./build-and-train-out")

import os, json
for root, _, files in os.walk("./build-and-train-out"):
    for f in sorted(files):
        print(" ", os.path.join(root, f))

with open("./build-and-train-out/output/result.json") as f:
    print("\nresult.json:", json.load(f))


---
## 9. **Deploy Locally with GGUF & Ollama**

Once your adapter has been downloaded with `client.download_model()`, you can take the model entirely off-cluster — merge it into the base, convert to **GGUF** (the format used by `llama.cpp`), and run it locally via **Ollama** with no GPU required.

Three steps:

1. **Merge** the LoRA adapter into the full base model weights
2. **Convert** the merged model to GGUF (optionally quantize to 4-bit)
3. **Deploy** via an Ollama Modelfile

> **Why GGUF?** Single-file format, runs on CPU or Apple Silicon, quantizes down to ~1/4 the size with minimal quality loss. Once you have a `.gguf` file, anyone with `ollama` installed can run your model with one command.


In [ ]:
# Step 1 — Merge LoRA adapter into the full base model
#
# PeftModel.merge_and_unload() fuses the adapter delta weights into the base
# model and returns a plain HuggingFace model ready for GGUF conversion.
#
# Note: §7 above already produced a merged model and reassigned MODEL_DIR to
# that location. This cell is idempotent — if it sees no adapter_config.json
# in MODEL_DIR (i.e. you already ran §7), it skips re-merging and just sets
# MERGED_DIR to the existing merged dir.

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch, os

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
MERGED_DIR = "./my-model-merged"

is_already_merged = not os.path.exists(os.path.join(MODEL_DIR, "adapter_config.json"))
if is_already_merged:
    print(f"MODEL_DIR={MODEL_DIR} is already a merged model (no adapter_config.json found).")
    print("Skipping re-merge and reusing it for GGUF conversion.")
    MERGED_DIR = MODEL_DIR
else:
    print("Loading base model in fp16...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    print("Attaching LoRA adapter & merging...")
    merged = PeftModel.from_pretrained(base, MODEL_DIR).merge_and_unload()

    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(MERGED_DIR)
    print(f"Merged model saved to: {MERGED_DIR}")


In [ ]:
# Step 2 — Convert merged model to GGUF and quantize to 4-bit
#
# Requires llama.cpp. The cell below clones and builds it if it's not already present.
# On Colab the whole thing takes ~3 minutes the first time, then is cached.

import os, subprocess

if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
    !pip install -q -r llama.cpp/requirements.txt
    !cmake -B llama.cpp/build llama.cpp && cmake --build llama.cpp/build --config Release -j --target llama-quantize

GGUF_F16 = "./my-model-f16.gguf"
GGUF_Q4  = "./my-model-q4_k_m.gguf"

# Convert merged HF model -> GGUF (fp16)
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_F16} --outtype f16

# Quantize fp16 GGUF -> 4-bit (Q4_K_M is the standard quality/size sweet spot)
!./llama.cpp/build/bin/llama-quantize {GGUF_F16} {GGUF_Q4} Q4_K_M

print(f"\nQuantized GGUF: {GGUF_Q4}")
!ls -lh {GGUF_Q4}

In [ ]:
# Step 3 — Deploy locally with Ollama
#
# Ollama wraps llama.cpp with a simple model registry and `ollama run` CLI.
# Install: https://ollama.com/download
#
# The Modelfile below tells Ollama which GGUF to load and what prompt template to use.

modelfile = '''FROM ./my-model-q4_k_m.gguf

TEMPLATE """Below is an instruction that describes a task.

### Instruction:
{{ .Prompt }}

### Response:
"""

PARAMETER temperature 0.7
PARAMETER stop "### Instruction:"
'''

with open("Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile written. To deploy locally, run on your machine:")
print()
print("    ollama create my-afrilink-model -f Modelfile")
print("    ollama run my-afrilink-model \"What is transfer learning?\"")
print()
print("Once registered, the model is reusable from any Ollama-compatible client")
print("(LM Studio, Open WebUI, the Ollama Python SDK, etc.) with no GPU required.")

---
## **Summary**

```
pip install afrilink-sdk

from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()                                         # API key from AFRILINK_API_KEY notebook secret

job = client.finetune(model="qwen2.5-0.5b", data=df, gpus=1)  # create job (runs on the OpenToken A100)
result = job.run(wait=True)                                    # submit & wait

client.download_model(result["job_id"], "./my-model")         # download adapter

```

For the full docs, check out our live on platform on https://dataspires.com and sign up for an account to use AfriLink today!